# Decode a real frame and call the pose model

The full run decodes complete annotated spans through Ambient. Here we replay one actual frame and make a fresh MediaPipe model call. GAVD frames are one-based; OpenCV frame positions are zero-based. Source videos remain under data/videos-gavd.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/restart_data.py').exists()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/restart_data.py').exists())
sys.path.insert(0, str(PROJECT))
from worldmodels.settings import configure
PROJECT, REPOSITORY = configure()
DATA = PROJECT/'artifacts/runs/restart-full10hz-v1'
RUN = PROJECT/'artifacts/runs/restart-dynamics-v1'
assert (DATA/'preparation.json').exists(), 'Run the full extraction command in README first.'
meta = pd.read_csv(DATA/'windows.csv')
with np.load(DATA/'windows.npz', allow_pickle=False) as z:
    x, valid = z['x'], z['valid']
print('Real GAVD observations:', x.shape, 'sources:', meta.source_id.nunique())


Select the first retained source in deterministic order. This is a pipeline example, not a favorable-result selection.

In [ ]:
from worldmodels.data import find_video
row=meta.sort_values(['source_id','sequence_id','first_frame']).iloc[0]
video=find_video(REPOSITORY,row.source_id,row.label)
assert video is not None
print('Source:',row.source_id,'annotation frame:',row.first_frame,'video:',video.relative_to(REPOSITORY))

Convert the frame index once and execute the shared image API. Missing joints remain missing.

In [ ]:
import cv2
from ambient.pose.keypoint_extractor import SequenceKeypointExtractor
cap=cv2.VideoCapture(str(video)); cap.set(cv2.CAP_PROP_POS_FRAMES,int(row.first_frame)-1)
ok,frame=cap.read();cap.release();assert ok
extractor=SequenceKeypointExtractor()
try:
    pose=extractor.extract_from_image(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB),str(REPOSITORY/'data/models/pose_landmarker_lite.task'))
finally:
    extractor.cleanup()
fig,ax=plt.subplots(figsize=(9,5));ax.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
points=[] if pose is None else [(k.x,k.y,k.confidence) for k in pose.keypoints]
if points:
    points=np.asarray(points); keep=points[:,2]>=.5
    ax.scatter(points[keep,0],points[keep,1],s=18,c='#00ccaa')
ax.axis('off');plt.show()
print('Fresh pose call completed; detected joints:',len(points))

Nominal sampling is 10 Hz. Actual frame numbers and decoder timestamps remain recorded; no interpolated coordinate is called observed.

In [ ]:
display(meta.filter(regex='time|fps|frame|horizon').head())
print((PROJECT/'docs/restart/data_contract.md').read_text().split('## Person selection')[0])

Inspect the source cache provenance and frame-level person-gate decisions. Pelvis containment does not prove identity in overlapping people.

In [ ]:
with np.load(DATA/'source_poses'/f'{row.source_id}.npz',allow_pickle=False) as raw:
    print('Raw cache fields:',raw.files)
gates=pd.read_csv(DATA/'person_gate_frames.csv')
display(gates.head())